# Limpieza y normalización
Limpia los comentarios de TikTok a partir de documento preprocesado que cuenta con la columna de etiquetado_humano que se encuentra en  `data/etiquetado_humano`. Los documentos limpios y normalizados serán enviado a la carpeta `data/limpieza_final`.

In [1]:
#Librerias de limpieza
import regex as re
import pandas as pd
from pathlib import Path
import os
import unicodedata
import ftfy

In [ ]:
#Extracción de rutas para la lectura y derivación de archivos
PROJECT_ROOT = Path(os.getcwd()).parent

# Ruta completa a carpetas
ETIQUETADO_PATH = PROJECT_ROOT / "data" / "etiquetado_humano" 
LIMPIEZA_PATH = PROJECT_ROOT / "data" / "limpieza_final"

In [3]:
#Reparación de encoding
def reparar_encoding(texto):
    if not isinstance(texto, str):
        return ""

    # 1. Reparar UTF-8 leído como latin1
    try:
        texto = texto.encode("latin1").decode("utf-8")
    except Exception:
        pass

    # 2. Reparar restos con ftfy
    texto = ftfy.fix_text(texto)

    # 3. Reemplazos residuales
    reemplazos = {
        "Ã¡": "á", "Ã©": "é", "Ã­": "í", "Ã\xad": "í",
        "Ã³": "ó", "Ã\x93": "Ó", "Ãº": "ú",
        "Ã±": "ñ", "Ã\x91": "Ñ",
        "Ã\x8d": "Í", "Ã\x81": "Á",
        "Â¿": "¿", "Â¡": "¡", "Â": "",
        "�": "",
    }

    for mal, bien in reemplazos.items():
        texto = texto.replace(mal, bien)

    # 4. Quitar controles raros, pero NO borrar emojis
    texto = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "", texto)

    return texto



In [4]:
#Funciones adicionales de limpieza estructural

#Reconecta palabras cortadas por guión al final de línea
def fix_hyphenation(texto: str) -> str:
    texto = re.sub(r"(\w)-\s*\n\s*(\w)", r"\1\2", texto)
    return texto

#Colapsar espacios en blanco o multilínea
def normalize_whitespace(texto):
    if not isinstance(texto, str):
        return ""
    texto = re.sub(r"\s+", " ", texto)

    return texto.strip()

In [5]:
EMOJIS_RISA = {"😂", "🤣", "😆", "😹", "😅", "😁", "😄", "😃"}

EMOJIS_ALEGRIA = {
    "😊", "🙂", "☺️", "😍", "🥰", "😘",
    "😻", "🤩", "😎", "😇", "😋", "😌",
    "❤️", "❤", "💖", "💕", "💜", "💙", "💚"
}

EMOJIS_TRISTEZA = {
    "😢", "😭", "☹️", "🙁", "😞",
    "😔", "😟", "🥲", "💔"
}

EMOJIS_ENOJO = {"😡", "🤬", "😠", "😤", "👿", "💢"}

EMOTICONOS = {
    r"(:\)+|:-\)+|=\)+)": "me gusta",
    r"(:d+|:-d+|xd+)": "me da risa",
    r"(:\(+|:-\(+|=\(+)": "me entristece",
    r"(>:\(+|d:<)": "me enoja",
}


def intensidad(contador, frase):
    if contador == 0:
        return ""
    elif contador == 1:
        return frase
    elif contador <= 3:
        return frase + " mucho"
    else:
        return frase + " muchisimo"


def reemplazar_emojis_repetidos(texto, emojis, frase):
    contador = 0

    for emoji in emojis:
        ocurrencias = texto.count(emoji)

        if ocurrencias > 0:
            contador += ocurrencias
            texto = texto.replace(emoji, " ")

    intensidad_texto = intensidad(contador, frase)

    if intensidad_texto:
        texto += " " + intensidad_texto + " "

    return texto


def limpiar_texto_es(texto):
    if not isinstance(texto, str):
        return ""

    texto = reparar_encoding(texto)
    texto = texto.lower()
    texto = unicodedata.normalize("NFKC", texto)

    texto = re.sub(r"\[sticker\]", " ", texto, flags=re.IGNORECASE)
    texto = re.sub(r"https?://\S+|www\.\S+", " ", texto)

    for patron, reemplazo in EMOTICONOS.items():
        texto = re.sub(patron, f" {reemplazo} ", texto, flags=re.IGNORECASE)

    texto = re.sub(r"\b(ja){2,}\b", " me da risa ", texto, flags=re.IGNORECASE)
    texto = re.sub(r"\b(ha){2,}\b", " me da risa ", texto, flags=re.IGNORECASE)
    texto = re.sub(r"\bjeje+\b", " me da risa ", texto, flags=re.IGNORECASE)
    texto = re.sub(r"\bjiji+\b", " me da risa ", texto, flags=re.IGNORECASE)
    texto = re.sub(r"\bxd+\b", " me da risa ", texto, flags=re.IGNORECASE)

    texto = reemplazar_emojis_repetidos(texto, EMOJIS_RISA, "me da risa")
    texto = reemplazar_emojis_repetidos(texto, EMOJIS_ALEGRIA, "me gusta")
    texto = reemplazar_emojis_repetidos(texto, EMOJIS_TRISTEZA, "me entristece")
    texto = reemplazar_emojis_repetidos(texto, EMOJIS_ENOJO, "me enoja")

    texto = re.sub(r"\p{Emoji_Presentation}|\p{Extended_Pictographic}", " ", texto)

    replacements = {
        "\u2018": "'",
        "\u2019": "'",
        "\u201c": '"',
        "\u201d": '"',
        "\u2013": "-",
        "\u2014": "-",
        "\u2026": " ",
        "\u00a0": " ",
        "\ufeff": "",
        "\u200b": "",
    }

    for old, new in replacements.items():
        texto = texto.replace(old, new)

    texto = re.sub(r"[\x00-\x1f\x7f]", " ", texto)

    # Conserva español, números y espacios
    texto = re.sub(r"[^a-záéíóúüñ0-9\s]", " ", texto)

    texto = re.sub(r"\s+", " ", texto).strip()

    return texto

In [6]:
#Recuperación de expresión de texto a partir de emojis
def solo_emojis(texto: str) -> bool:
    """
    Retorna True si el texto contiene solo emojis (y caracteres invisibles como ZWJ o VS16),
    y False si hay algún otro carácter (letras, números, espacios, puntuación, etc.).
    """
    # Patrón que coincide con cualquier emoji (incluyendo secuencias complejas)
    patron_emoji = re.compile(r'\p{Emoji_Presentation}|\p{Extended_Pictographic}')
    
    # Eliminar posibles caracteres de control/unión que no son emojis por sí mismos
    texto_limpio = re.sub(r'[\u200d\uFE0F\uFE0E\u061C\u200E\u200F\u202A-\u202E]', '', texto)
    
    if not texto_limpio:
        return False
    
    # Buscar todos los "trozos" que no son emojis (caracteres normales)
    resto = re.sub(patron_emoji, '', texto_limpio)
    
    # Si después de quitar emojis queda algo, hay caracteres no-emoji
    return len(resto.strip()) == 0



def remove_special_characters(text: str) -> str:
    """Elimina caracteres de control, basura Unicode, emojis y etiquetas [STICKER], preservando acentos y ñ."""
    
    if not isinstance(text, str):
        return text
    
    # 1. Eliminar etiquetas [STICKER] (NUEVO)
    # Esto elimina [STICKER] exacto, insensible a mayúsculas/minúsculas
    #text = re.sub(r'\[STICKER\]', '', text, flags=re.IGNORECASE)
    # También puedes eliminar variantes como [Sticker], [sticker], etc. (ya cubierto con IGNORECASE)
    
    # También eliminar si hay espacios alrededor (opcional)
    # text = re.sub(r'\s*\[STICKER\]\s*', ' ', text, flags=re.IGNORECASE)
    
    # 2. Eliminar caracteres de control (excepto newlines y tabs)
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "", text)
    
    # 3. Reemplazar caracteres Unicode problemáticos comunes
    replacements = {
        "\u2018": "'", "\u2019": "'",  # Comillas simples tipográficas
        "\u201c": '"', "\u201d": '"',  # Comillas dobles tipográficas
        "\u2013": "-", "\u2014": "-",  # Guiones em/en
        "\u2026": "...",               # Elipsis
        "\u00a0": " ",                 # Non-breaking space
        "\ufeff": "",                  # BOM
        "\u200b": "",                  # Zero-width space
        "\uf0b7": "- ",                # Bullet point (symbol font)
        "\uf0a7": "- ",                # Otro bullet
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    
    # 4. Eliminar emojis
    # Esta regex captura la mayoría de emojis, incluyendo:
    # - Emojis básicos (😀, ❤️)
    # - Emojis con modificadores de tono de piel (👋🏽)
    # - Emojis ZWJ (familia, profesiones: 👨‍👩‍👧‍👦, 👩‍💻)
    # - Símbolos de flechas y otros (⚠️, ㊗️)
    # - Números y letras rodeados (▶️, ℹ️)
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # Emojis emociones
        "\U0001F300-\U0001F5FF"  # Símbolos y pictogramas
        "\U0001F680-\U0001F6FF"  # Transporte y mapas
        "\U0001F700-\U0001F77F"  # Símbolos alquímicos
        "\U0001F780-\U0001F7FF"  # Símbolos geométricos extendidos
        "\U0001F800-\U0001F8FF"  # Flechas suplementarias-C
        "\U0001F900-\U0001F9FF"  # Emojis suplementarios (2020+)
        "\U0001FA00-\U0001FA6F"  # Ajedrez y símbolos extendidos
        "\U0001FA70-\U0001FAFF"  # Emojis adicionales (2021+)
        "\U00002702-\U000027B0"  # Símbolos dingbat
        "\U000024C2-\U0001F251"  # Símbolos encerrados
        "]+",
        flags=re.UNICODE
    )
    
    # También capturar emojis con modificadores ZWJ (familias, etc.)
    # Eliminamos primero los que tienen joiners y variantes
    text = re.sub(r'[\U0001F3FB-\U0001F3FF]', '', text)  # Tono de piel
    text = re.sub(r'\u200D', '', text)  # Zero-width joiner
    text = emoji_pattern.sub(r'', text)
    
    # 5. Limpieza final: eliminar espacios múltiples que puedan quedar
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text



def indices(datos: str) -> str:
    #limpiamos celdas vacias
    borrar = []
    for i in range(len(datos)):
        if ((type(datos[i]) != str)):
            borrar.append(i)
        elif solo_emojis(datos[i]):
            borrar.append(i)
        elif (datos[i] == '[Sticker]' or datos[i] == '[Sticker] '):
            borrar.append(i)
        elif (datos[i] == ''):
            borrar.append(i)
    return borrar

In [7]:
archivo = ['infraestructura_etiquetado_humano','turismo_etiquetado_humano','seguridad_etiquetado_humano']
archivo_fin = ['infraestructura','turismo','seguridad']
for i in range(len(archivo)):
    aux = ETIQUETADO_PATH / f"{archivo[i]}.csv"
    #print(aux)
    datos = pd.read_csv(aux, encoding="latin-1")

    datos.columns = (
    datos.columns
    .str.replace("ï»¿", "", regex=False)
    .str.strip()
    .str.lower()
    )

    datos = datos.loc[:, ~datos.columns.str.contains("^unnamed", case=False)]

    print(datos.columns.tolist())
    #print(datos.head())
    
    
    datos_limpios = datos.drop(indices(datos['comentario']))
    datos_limpios["comentario"] = (
        datos_limpios["comentario"]
        .astype(str)
        .apply(reparar_encoding)
        .apply(fix_hyphenation)
        .apply(normalize_whitespace)
        .apply(limpiar_texto_es)
    )
    
    aux2 = LIMPIEZA_PATH / f"{archivo_fin[i]}_limpio.csv"
    #print(aux2)
    datos_limpios.to_csv(aux2, encoding = 'utf-8')
print('Limpieza de comentarios terminada')

['comentario', 'etiquetado_humano']
['comentario', 'rango_humano']
['comentario', 'rango_humano']
Limpieza de comentarios terminada
